🚨 Drug Recall Risk Classifier — FDA Class I / II / III System
FDA aur CDSCO drug recalls ko 3 classes mein divide karte hain based on risk level. Ek tool banao jo automatically recall class assign kare.

Class banao DrugRecall — attributes: drug_name, batch_id, reason, defect_type, units_affected.

Method banao classify() — defect_type ke hisaab se class assign karo:
— "contamination" ya "wrong_drug" → Class I (life threatening)
— "mislabeling" ya "subpotent" → Class II (may cause harm)
— "packaging_defect" → Class III (unlikely to cause harm)

Method banao recall_action() — Class I = "⛔ Immediate withdrawal", Class II = "⚠ Urgent recall", Class III = "📦 Market withdrawal".

Custom exception UnknownDefectError — agar defect_type known nahi hai.
Method save_recall() — "recall_log.txt" mein save karo.

5 recalls test karo — ek unknown defect bhi daalo.

In [1]:

import logging
from datetime import date

logging.basicConfig(
    filename="recall_system_log.txt",
    level=logging.DEBUG,
    format="%(asctime)s — %(levelname)s — %(message)s"
)


class UnknownDefectError(Exception):
    pass


class DrugRecall:
    def __init__(self,drug_name, batch_id, reason, defect_type, units_affected):
        self.drug_name = drug_name
        self.batch_id = batch_id
        self.reason = reason
        self.defect_type = defect_type
        self.units_affected = units_affected

    def classify(self):
        if self.defect_type == "contamination" or self.defect_type == "wrong_drug":
            return "Class I"
        elif self.defect_type == "mislabeling" or self.defect_type == "subpotent":
            return "Class II"
        elif self.defect_type == "packaging_defect":
            return "Class III"
        else:
            raise UnknownDefectError

    def recall_action(self):
        if self.classify() == "Class I":
            return "Immediate withdrawal"
        elif self.classify() == "Class II":
            return " Urgent recall"
        elif self.classify() == "Class III":
            return " Market withdrawal"

    def save_recall(self):
        c = self.classify()
        act = self.recall_action()
        with open("recall_log.txt", "a") as f:
            f.write(f"Date   : {date.today()}\n")
            f.write(f"Drug   : {self.drug_name}\n")
            f.write(f"Batch  : {self.batch_id}\n")
            f.write(f"Reason : {self.reason}\n")
            f.write(f"Defect : {self.defect_type}\n")
            f.write(f"Class  : {c}\n")
            f.write(f"Action : {act}\n")
            f.write(f"Units  : {self.units_affected:,}\n")
            f.write("-" * 45 + "\n")
        logging.info(f"{self.drug_name} | {self.batch_id} | {c}")

    #this function is to call so through this all other func will operate simultyanously
    def process(self):
        print(f"\n{'='*50}")
        try:
            c = self.classify()
            act = self.recall_action()
            print(f"Drug | {self.drug_name}")
            print(f"Batch | {self.batch_id}")
            print(f"Reason | {self.reason}")
            print(f"Defect | {self.defect_type}")
            print(f"Class | {c}")
            print(f"Action | {act}")
            print(f"Units | {self.units_affected:,}")
            self.save_recall()  #we hae to call this and this contain all data if failed to call it will not crash and moved to exception part.
            
        except UnknownDefectError as e:
            print(f"ERROR: {e}")
            logging.warning(f"{self.drug_name}-{self.batch_id} UNKNOWN DEFECT FOUND - {e}")

    def __str__(self):
        return f"{self.drug_name} | {self.batch_id}"

#lets try wth random AI generated examples.
recalls = [
    DrugRecall("Metformin 500mg", "BT-001",
               "Microbial contamination found",
               "contamination", 50000),         

    DrugRecall("Aspirin 75mg", "BT-002",
               "Dosage label incorrect",
               "mislabeling", 25000),           

    DrugRecall("Insulin 100IU", "BT-003",
               "Wrong drug in packaging",
               "wrong_drug", 10000),            

    DrugRecall("Atorvastatin 10mg", "BT-004",
               "Blister pack seal broken",
               "packaging_defect", 15000),      
    DrugRecall("Amoxicillin 250mg", "BT-005",
               "Unknown issue reported",
               "colour_change", 5000),         
]

for recall in recalls:
    recall.process()        



Drug | Metformin 500mg
Batch | BT-001
Reason | Microbial contamination found
Defect | contamination
Class | Class I
Action | Immediate withdrawal
Units | 50,000

Drug | Aspirin 75mg
Batch | BT-002
Reason | Dosage label incorrect
Defect | mislabeling
Class | Class II
Action |  Urgent recall
Units | 25,000

Drug | Insulin 100IU
Batch | BT-003
Reason | Wrong drug in packaging
Defect | wrong_drug
Class | Class I
Action | Immediate withdrawal
Units | 10,000

Drug | Atorvastatin 10mg
Batch | BT-004
Reason | Blister pack seal broken
Defect | packaging_defect
Class | Class III
Action |  Market withdrawal
Units | 15,000

ERROR: 
